In [1]:
%pip install ta scikit-learn pandas numpy

import pandas as pd
import numpy as np
import os
import ta
from sklearn.preprocessing import StandardScaler

SYMBOL = 'XRPUSDT'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()

SEQ_LENGTH = 60          # 5h of 5m candles fed into the LSTM
VOL_WINDOW = 12          # predict realized volatility over the NEXT 12 candles (1 hour)
PREDICT_AHEAD = 1
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
STEP = 3

FEATURE_COLS = [
    'log_ret', 'rsi', 'rsi_change', 'rsi_accel',
    'macd', 'macd_slope',
    'bb_pband', 'bb_pband_change',
    'volume_z', 'vol_spike',
    'ma_dist', 'atr_norm',
    'ema_ratio_fast', 'ema_ratio_slow',
    'adx',
    'hour_sin', 'hour_cos'
]

# TARGET: future realized volatility (std of log returns over the next VOL_WINDOW candles).
# BASELINE: current realized volatility (persistence) - the number the model must beat.
TARGET_COL = 'target_vol'
BASELINE_COL = 'current_vol'


class DataPreprocessorLSTM:
    def load_and_clean_data(self, filepath):
        print(f"Loading data from {filepath}...")
        df = pd.read_csv(filepath)
        df['open_time'] = pd.to_datetime(df['open_time'])
        df = df.sort_values('open_time').drop_duplicates(subset=['open_time']).ffill().dropna()

        # Correct log return formula (no arbitrary multiplier inside the log)
        df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

        # RSI features
        df['rsi'] = ta.momentum.rsi(df['close'], window=14) / 100.0
        df['rsi_change'] = df['rsi'].diff(periods=3)
        df['rsi_accel'] = df['rsi_change'].diff(periods=2)

        # MACD features
        macd = ta.trend.MACD(df['close'])
        macd_raw = macd.macd_diff()
        df['macd'] = (macd_raw - macd_raw.rolling(window=100).mean()) / (macd_raw.rolling(window=100).std() + 1e-9)
        df['macd_slope'] = macd_raw.diff(periods=2)

        # Bollinger Bands - level and change
        bb = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
        df['bb_pband'] = bb.bollinger_pband()
        df['bb_pband_change'] = df['bb_pband'].diff(periods=1)

        # Volume features
        vol_log = np.log(df['volume'] + 1)
        vol_ma = vol_log.rolling(window=20).mean()
        vol_std = vol_log.rolling(window=20).std()
        df['volume_z'] = (vol_log - vol_ma) / (vol_std + 1e-9)
        df['vol_spike'] = (df['volume_z'] > 2.0).astype(float)

        # MA distance (no arbitrary multiplier - scaler handles normalization)
        df['ma_20'] = df['close'].rolling(window=20).mean()
        df['ma_dist'] = (df['close'] - df['ma_20']) / (df['ma_20'] + 1e-9)

        # ATR normalized: current volatility relative to price
        atr = ta.volatility.AverageTrueRange(df['high'], df['low'], df['close'], window=14)
        df['atr_norm'] = atr.average_true_range() / (df['close'] + 1e-9)

        # EMA cross ratios: fast and slow trend signals
        ema_8 = df['close'].ewm(span=8, adjust=False).mean()
        ema_21 = df['close'].ewm(span=21, adjust=False).mean()
        ema_55 = df['close'].ewm(span=55, adjust=False).mean()
        df['ema_ratio_fast'] = (ema_8 - ema_21) / (ema_21 + 1e-9)
        df['ema_ratio_slow'] = (ema_21 - ema_55) / (ema_55 + 1e-9)

        # ADX normalized to [0, 1]
        adx_ind = ta.trend.ADXIndicator(df['high'], df['low'], df['close'], window=14)
        df['adx'] = adx_ind.adx() / 100.0

        # Hour encoding
        df['hour'] = df['open_time'].dt.hour
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

        # === VOLATILITY TARGET ===
        # current_vol = realized volatility over the PAST VOL_WINDOW candles (known at prediction time).
        # target_vol  = realized volatility over the NEXT VOL_WINDOW candles (what we predict).
        # Predicting current_vol forward (persistence) is the baseline we want to beat.
        df['current_vol'] = df['log_ret'].rolling(window=VOL_WINDOW).std()
        df['target_vol'] = df['current_vol'].shift(-VOL_WINDOW)

        # Clip extreme volatility spikes at the 99.5th percentile to stabilise training
        cap = df['target_vol'].quantile(0.995)
        df['target_vol'] = df['target_vol'].clip(upper=cap)

        df = df.dropna()
        print(f"  Rows after cleaning: {len(df)}")
        print(f"  Target (future vol) stats: mean={df['target_vol'].mean():.6f}, std={df['target_vol'].std():.6f}")
        print(f"  Corr(current_vol, future_vol) = {df['current_vol'].corr(df['target_vol']):.3f}  (persistence baseline strength)")
        return df

    def create_sequences(self, data, target, baseline, seq_length, step=1):
        xs, ys, bs = [], [], []
        for i in range(0, len(data) - seq_length - PREDICT_AHEAD + 1, step):
            xs.append(data[i : i + seq_length])
            ys.append(target[i + seq_length + PREDICT_AHEAD - 1])
            bs.append(baseline[i + seq_length + PREDICT_AHEAD - 1])
        return np.array(xs), np.array(ys), np.array(bs)

    def process(self):
        data_path = os.path.join(BASE_DIR, 'data', f'{SYMBOL}_5m_data.csv')
        df = self.load_and_clean_data(data_path)

        data = df[FEATURE_COLS].values
        target = df[TARGET_COL].values
        baseline = df[BASELINE_COL].values

        n = len(data)
        train_end = int(n * TRAIN_SPLIT)
        val_end = int(n * (TRAIN_SPLIT + VAL_SPLIT))

        scaler = StandardScaler()
        train_data_scaled = scaler.fit_transform(data[:train_end])
        val_data_scaled = scaler.transform(data[train_end:val_end])
        test_data_scaled = scaler.transform(data[val_end:])

        X_train, y_train, b_train = self.create_sequences(train_data_scaled, target[:train_end], baseline[:train_end], SEQ_LENGTH, STEP)
        X_val,   y_val,   b_val   = self.create_sequences(val_data_scaled, target[train_end:val_end], baseline[train_end:val_end], SEQ_LENGTH, STEP)
        X_test,  y_test,  b_test  = self.create_sequences(test_data_scaled, target[val_end:], baseline[val_end:], SEQ_LENGTH, STEP)

        save_dir = os.path.join(BASE_DIR, 'processed_data_lstm', SYMBOL)
        os.makedirs(save_dir, exist_ok=True)

        np.save(os.path.join(save_dir, 'X_train.npy'), X_train)
        np.save(os.path.join(save_dir, 'y_train.npy'), y_train)
        np.save(os.path.join(save_dir, 'X_val.npy'), X_val)
        np.save(os.path.join(save_dir, 'y_val.npy'), y_val)
        np.save(os.path.join(save_dir, 'X_test.npy'), X_test)
        np.save(os.path.join(save_dir, 'y_test.npy'), y_test)
        # Persistence baseline (current vol) aligned with each target - used in the trainer for comparison
        np.save(os.path.join(save_dir, 'b_train.npy'), b_train)
        np.save(os.path.join(save_dir, 'b_val.npy'), b_val)
        np.save(os.path.join(save_dir, 'b_test.npy'), b_test)

        print(f"\n✅ Preprocessing Complete for {SYMBOL} (LSTM volatility target). Saved to {save_dir}")
        print(f"   X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")
        print(f"   y_train (future vol) mean={y_train.mean():.6f}, std={y_train.std():.6f}")


if __name__ == "__main__":
    DataPreprocessorLSTM().process()


Mounted at /content/drive
Loading data from /content/drive/MyDrive/CryptoProject/data/XRPUSDT_5m_data.csv...
  Rows after cleaning: 841726
  Target (future vol) stats: mean=0.002316, std=0.001974
  Corr(current_vol, future_vol) = 0.670  (persistence baseline strength)

✅ Preprocessing Complete for XRPUSDT (LSTM volatility target). Saved to /content/drive/MyDrive/CryptoProject/processed_data_lstm/XRPUSDT
   X_train=(224440, 60, 17), X_val=(28038, 60, 17), X_test=(28038, 60, 17)
   y_train (future vol) mean=0.002356, std=0.002031
